In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)
from pyspark.sql.functions import col, round, sum, avg, count, upper,initcap,desc

schema = StructType([
    StructField("order_id",   IntegerType(), True),
    StructField("customer",   StringType(),  True),
    StructField("city",       StringType(),  True),
    StructField("category",   StringType(),  True),
    StructField("amount",     DoubleType(),  True),
    StructField("order_date", StringType(),  True),
    StructField("quantity",   IntegerType(), True),
])

data = [
  (1,  "Alice",   "chicago",  "Electronics", 1200.50, "2024-01-15", 2),
  (2,  "Bob",     "New York", "clothing",      89.99, "2024-01-16", 1),
  (3,  "Carol",   "CHICAGO",  "Electronics",  450.00, "2024-01-17", 3),
  (4,  "David",   "Houston",  "Food",           23.49, "2024-01-17", 1),
  (5,  "Eve",     "New York", "Electronics",  999.99, "2024-01-18", 1),
  (6,  "Frank",   "Chicago",  "clothing",     199.00, "2024-01-19", 2),
  (7,  "Grace",   "houston",  "Food",           55.00, "2024-01-20", 4),
  (8,  "Henry",   "New York", "Electronics", 2300.00, "2024-01-20", 1),
  (9,  "Ivy",     "Chicago",  "Food",           18.75, "2024-01-21", 2),
  (10, "James",   "houston",  "clothing",     349.00, "2024-01-21", 3),
  (11, "Karen",   "Chicago",  "Electronics",  870.00, "2024-01-22", 1),
  (12, "Leo",     "New York", None,            125.00, "2024-01-22", 2),
  (13, "Maya",    "Chicago",  "Food",           67.50, "2024-01-23", 1),
  (14, "Nathan",  "Houston",  "Electronics",  540.00, "2024-01-23", 2),
  (15, "Olivia",  "New York", "clothing",     210.00, "2024-01-24", 1),
]

columns = ["order_id", "customer", "city", "category",
           "amount", "order_date", "quantity"]

df = spark.createDataFrame(data, schema)
df.printSchema()
df.show()



In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_bronze")

print("retail_bronze table created")

In [0]:


df_cleaned = (
    df
    .withColumn("city", initcap(col("city")))
    .withColumn("category", initcap(col("category")))
    .fillna(value="Unknown",subset=["category"])
    .withColumn("total_value", col("amount") * col("quantity"))
    .filter(col("amount") >= 20)
    .select("order_id", "customer", "city", "category", "amount", "quantity","total_value", "order_date")
)

df_cleaned.show()

df_cleaned.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver")

print("retail_silver table created")

In [0]:
df_gold_city = df_cleaned.groupBy("city").agg(
    round(sum("amount"),2).alias("total_revenue"),
    count("order_id").alias("total_orders"),
    round(avg("total_value"),2).alias("avg_order_value")
).sort(desc("total_revenue"))

df_gold_city.show()

df_gold_city.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_gold_city")

print("retail_gold_city table created")

In [0]:
df_gold_category = df_cleaned.groupBy("category").agg(
    round(sum("amount"),2).alias("total_revenue"),
    count("order_id").alias("total_orders"),
    round(avg("total_value"),2).alias("avg_order_value")
).sort(desc("total_revenue"))

df_gold_category.show()

df_gold_category.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_gold_category")

print("retail_gold_category table created")

In [0]:
%sql

-- Q1: highest revenue city
SELECT city, ROUND(SUM(amount), 2) AS total_revenue
FROM retail_silver GROUP BY city
ORDER BY total_revenue DESC LIMIT 1

In [0]:
%sql
-- Q2: customer who spent the most
SELECT customer, ROUND(SUM(amount), 2) AS total_spent
FROM retail_silver GROUP BY customer
ORDER BY total_spent DESC LIMIT 1

In [0]:
%sql
-- Q3: top category in Chicago
SELECT category, ROUND(SUM(amount), 2) AS total_revenue
FROM retail_silver WHERE city = "Chicago"
GROUP BY category ORDER BY total_revenue DESC LIMIT 1


In [0]:
%sql
select * from retail_silver
order by total_value desc
limit 3

In [0]:
%sql
select 
  count(case when amount >= 500 then order_id end) as orders_above_500,
  count(case when amount < 500 then order_id end) as orders_below_500
from retail_silver